# Steering-independence — Colab runner (Goodfire follow-up 2.1)

Runs `june/steering_independence` end-to-end and produces the two headline artifacts:

1. **Off-target steering matrix** — steer along trait *i*'s difference-of-means vector,
   judge the output on every trait *j* (behavioral transfer), vs the **geometric cosine**
   of the steering vectors. Tests the paper's off-target-steering claim (Fig 9).
2. **Novel heuristic** — `scatter_geo_vs_beh.png`: does vector cosine *predict* behavioral
   spillover? Reported with Pearson/Spearman + a **random-direction noise floor** and a
   residual plot (coupling beyond geometry).

**Conventions** (`june/CLAUDE.md`): GPU work on Colab; the repo is the **Drive-resident
clone** pulled `--ff-only`; secrets come from Colab `userdata`; results are written to
**Drive, never git**. A100 for Llama-8B; L4/T4 fine for Qwen3-4B.

> Note: this replaces the older `steering_independence.ipynb`, whose bootstrap points at
> `niels/propensities` (stale — the layout moved to `niels/`) and `pip install -e` a missing
> dir. This notebook puts `niels/` on the path instead.

## 1 · Bootstrap (Colab: Drive clone + deps + secrets + paths)

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    REPO = Path("/content/drive/MyDrive/spar-ood-propensities")
    if not REPO.exists():
        subprocess.run(["git","clone","https://github.com/nielsrolf/spar-ood-propensities",str(REPO)],check=True)
    # Pull latest so the runtime has your pushed commit BEFORE anything runs (stale-clone guard)
    print(subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],capture_output=True,text=True).stdout)
    subprocess.run(["pip","-q","install","transformers","accelerate","torch","openai",
                    "seaborn","pyyaml","tqdm","scipy","bitsandbytes","scikit-learn",
                    "python-dotenv","pandas","matplotlib"],check=True)
    # Secrets from Colab userdata (never hardcode)
    for k in ("OPENROUTER_API_KEY","HF_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: print("WARN: secret", k, "not set in Colab userdata")
    # Results to Drive, OUTSIDE the git tree:
    RESULTS = Path("/content/drive/MyDrive/spar/steering_independence"); RESULTS.mkdir(parents=True,exist_ok=True)
else:
    REPO = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd().parents[1]
    RESULTS = REPO/"june"/"steering_independence"/"outputs"

STEER = REPO/"june"/"steering_independence"
os.chdir(STEER)
# CORRECT import roots: niels/ (so `experiments.eval_config` resolves) + the steering dir.
for p in (str(REPO/"niels"), str(STEER)):
    if p not in sys.path: sys.path.insert(0, p)
print("cwd:", os.getcwd(), "\nresults ->", RESULTS)

## 2 · Stale-clone / data-shape guard (fail loud before any GPU work)

In [ ]:
from trait_registry import ALL_TRAITS, load_contrastive_pairs
counts = {t: len(load_contrastive_pairs(t, "train")) for t in ALL_TRAITS}
print("\n".join(f"  {t:22s} train_pairs={n}" for t,n in counts.items()))
assert all(n > 0 for n in counts.values()), "Missing contrastive pairs — check the niels/ path / repo pull"
print(f"\nOK: {len(ALL_TRAITS)} traits, {sum(counts.values())} train pairs")

## 3 · Config (Qwen3-4B `config.yaml` or Llama-8B `config_llama8b.yaml`)

In [ ]:
import yaml
CONFIG_FILE = "config.yaml"   # or "config_llama8b.yaml"
with open(CONFIG_FILE) as f: config = yaml.safe_load(f)
config["output_dir"] = str(RESULTS)   # results to Drive, non-git
print("model:", config["model_id"], "| judge:", config["judge"]["model"],
      "| coherence_threshold:", config.get("coherence_threshold"),
      "| n_random_controls:", config["behavioral"]["n_random_controls"])

## 4 · Step 1 — extract difference-of-means steering vectors

In [ ]:
from extract_vectors import extract_all
meta = extract_all(config)
for t,i in meta.items(): print(f"{t}: {i['n_layers']} layers, dim={i['hidden_dim']}, {i['n_pairs']} pairs")

## 5 · Step 2 — geometric similarity (cosine) matrix

In [ ]:
from geometric_similarity import compute_and_save
import seaborn as sns, matplotlib.pyplot as plt
geo_df = compute_and_save(config)
fig,ax = plt.subplots(figsize=(9,7))
sns.heatmap(geo_df, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, ax=ax)
ax.set_title("Geometric similarity of steering vectors (per-trait best layer)"); plt.tight_layout(); plt.show()

## 6 · Step 3 — steered generation (GPU) + random-direction controls

In [ ]:
from behavioral_steering import generate_all
gen = generate_all(config)   # writes generations/<src>_to_<tgt>.jsonl + random controls
print("generated", sum(gen.values()), "responses across", len(gen), "files")

## 7 · Behavioral transfer matrix (judge = gpt-4o-mini via OpenRouter)

In [ ]:
from behavioral_steering import judge_all
import seaborn as sns, matplotlib.pyplot as plt
beh_df = await judge_all(config)
fig,ax = plt.subplots(figsize=(9,7))
sns.heatmap(beh_df, annot=True, fmt=".1f", cmap="RdBu_r", center=0, square=True, ax=ax)
ax.set_title("Behavioral transfer Δ (steer source → measure target)"); plt.tight_layout(); plt.show()

## 8 · Coherence gate (cells below `coherence_threshold`=70 are masked)

In [ ]:
from behavioral_steering import judge_coherence
coh_df = await judge_coherence(config)
import seaborn as sns, matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize=(10,7))
sns.heatmap(coh_df, annot=True, fmt=".0f", cmap="RdYlGn", vmin=0, vmax=100, ax=ax)
ax.set_title("Mean coherence by source × target (gate any cell < 70)"); plt.tight_layout(); plt.show()

## 9 · Step 4 — HEURISTIC: does cosine predict behavioral spillover?
Produces `scatter_geo_vs_beh.png` (Pearson/Spearman + random-direction noise floor) and a
residual plot. This is the core test for follow-up 2.1.

In [ ]:
import compare_and_plot
out = compare_and_plot.run(config)
print("saved:", sorted((RESULTS/'plots').glob('*.png')) if (RESULTS/'plots').exists() else out)
from IPython.display import Image, display
p = RESULTS/"plots"/"scatter_geo_vs_beh.png"
if p.exists(): display(Image(str(p)))

## 10 · Robustness — re-run the heuristic excluding sycophancy (known outlier)

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable,"reanalyse.py","--exclude","sycophancy"],
                     capture_output=True,text=True,cwd=str(STEER)).stdout)

## 11 · Pull back for local analysis
Results are on Drive at `MyDrive/spar/steering_independence/` (matrices/, plots/, generations/).
Download `matrices/*.csv` + `plots/scatter_geo_vs_beh.png` and analyse locally — compare the
**geometric→behavioral** coupling here against the **train→eval** spillover from wave-1 (do the
same self-elevation traits dominate?). Do NOT commit generations/ or model outputs to git.